In [1]:
import pandas as pd

In [2]:
df = pd.read_csv("shop_smart_ecommerce.csv")

In [3]:
df.columns

Index(['Administrative', 'Administrative_Duration', 'Informational',
       'Informational_Duration', 'ProductRelated', 'ProductRelated_Duration',
       'BounceRates', 'ExitRates', 'PageValues', 'SpecialDay', 'Month',
       'OperatingSystems', 'Browser', 'Region', 'TrafficType', 'VisitorType',
       'Weekend', 'Revenue'],
      dtype='object')

In [4]:
df['Revenue'].value_counts()

Revenue
False    10422
True      1908
Name: count, dtype: int64

In [5]:
df.groupby('Revenue')['Informational'].mean()
df.groupby('Revenue')['Informational_Duration'].mean()

Revenue
False    30.236237
True     57.611427
Name: Informational_Duration, dtype: float64

In [6]:
pd.crosstab(df['OperatingSystems'], df['Revenue'])

Revenue,False,True
OperatingSystems,,
1,2206,379
2,5446,1155
3,2287,268
4,393,85
5,5,1
6,17,2
7,6,1
8,62,17


In [7]:
df.groupby('OperatingSystems')['Revenue'].mean()

OperatingSystems
1    0.146615
2    0.174973
3    0.104892
4    0.177824
5    0.166667
6    0.105263
7    0.142857
8    0.215190
Name: Revenue, dtype: float64

In [8]:
from sklearn.model_selection import train_test_split

X = df.drop('Revenue',axis=1)
y = df['Revenue']

X_train,X_test,y_train,y_test = train_test_split(
    X,y,test_size=0.2,random_state=42,
)
X_columns=X_train.columns

In [9]:
cat_cols = ["OperatingSystems","Browser","VisitorType","Month","Region","TrafficType","Weekend"]
num_cols = []
for i in X_columns:
    if i not in cat_cols:
        num_cols.append(i)

In [10]:
from sklearn.preprocessing import OneHotEncoder
ohe = OneHotEncoder(drop="first",sparse_output=False,handle_unknown="ignore")
X_train_cat = ohe.fit_transform(X_train[cat_cols])
X_test_cat = ohe.transform(X_test[cat_cols])

X_train_num = X_train[num_cols]
X_test_num = X_test[num_cols]

In [11]:
X_train_cat_df = pd.DataFrame(
    X_train_cat,
    columns = ohe.get_feature_names_out(cat_cols),
    index = X_train.index
)

X_test_cat_df = pd.DataFrame(
    X_test_cat,
    columns = ohe.get_feature_names_out(cat_cols),
    index = X_test.index
)

In [12]:
X_train_final = pd.concat([X_train_num,X_train_cat_df],axis=1)
X_test_final = pd.concat([X_test_num,X_test_cat_df],axis=1)

In [13]:
X_train_final.shape
X_test_final.shape

(2466, 68)

In [14]:
from sklearn.tree import DecisionTreeClassifier
dt = DecisionTreeClassifier(random_state=42)

dt.fit(X_train_final,y_train)

,criterion,'gini'
,splitter,'best'
,max_depth,None
,min_samples_split,2
,min_samples_leaf,1
,min_weight_fraction_leaf,0.0
,max_features,None
,random_state,42
,max_leaf_nodes,None
,min_impurity_decrease,0.0
,class_weight,None


In [15]:
y_train_pred = dt.predict(X_train_final)
y_test_pred = dt.predict(X_test_final)

In [16]:
from sklearn.metrics import f1_score

train_f1 = f1_score(y_train,y_train_pred)
test_f1 = f1_score(y_test,y_test_pred)

print("Train F1: ",train_f1)
print("Test F1: ",test_f1)

Train F1:  1.0
Test F1:  0.5493975903614458


In [17]:
dt.get_depth()

30

In [18]:
dt.get_n_leaves()

np.int64(795)

In [25]:
max_depth = [2,3,4,5,6,7,8,9,10,11,12]

for depth in max_depth:
    dt = DecisionTreeClassifier(random_state=42,max_depth=depth)
    dt.fit(X_train_final,y_train)

    y_train_pred = dt.predict(X_train_final)
    y_test_pred = dt.predict(X_test_final)

    train_f1 = f1_score(y_train,y_train_pred)
    test_f1 = f1_score(y_test,y_test_pred)

    print(f"Train F1 for depth {depth}: {train_f1}")
    print(f"Test F1 for depth {depth} : {test_f1}")
    print("----------------------------------------")

Train F1 for depth 2: 0.5737571312143439
Test F1 for depth 2 : 0.5256609642301711
----------------------------------------
Train F1 for depth 3: 0.6570116566584245
Test F1 for depth 3 : 0.6357615894039735
----------------------------------------
Train F1 for depth 4: 0.6676374226428832
Test F1 for depth 4 : 0.648
----------------------------------------
Train F1 for depth 5: 0.6661490683229814
Test F1 for depth 5 : 0.5973645680819912
----------------------------------------
Train F1 for depth 6: 0.7063579566335906
Test F1 for depth 6 : 0.6284916201117319
----------------------------------------
Train F1 for depth 7: 0.7275426874536006
Test F1 for depth 7 : 0.6237762237762238
----------------------------------------
Train F1 for depth 8: 0.7632850241545893
Test F1 for depth 8 : 0.6141078838174274
----------------------------------------
Train F1 for depth 9: 0.7973124300111982
Test F1 for depth 9 : 0.592797783933518
----------------------------------------
Train F1 for depth 10: 0.83901

In [27]:
split_values = [2, 5, 10, 20, 50, 100]

for s in split_values:
    dt = DecisionTreeClassifier(random_state=42,min_samples_split=s)
    dt.fit(X_train_final,y_train)

    y_train_pred = dt.predict(X_train_final)
    y_test_pred = dt.predict(X_test_final)

    train_f1 = f1_score(y_train,y_train_pred)
    test_f1 = f1_score(y_test,y_test_pred)

    print(f"Train F1 for mss {s}: {train_f1}")
    print(f"Test F1 for mss {s} : {test_f1}")
    print("----------------------------------------")
    

Train F1 for mss 2: 1.0
Test F1 for mss 2 : 0.5493975903614458
----------------------------------------
Train F1 for mss 5: 0.9576127500847745
Test F1 for mss 5 : 0.565483476132191
----------------------------------------
Train F1 for mss 10: 0.8947184330185379
Test F1 for mss 10 : 0.5710659898477157
----------------------------------------
Train F1 for mss 20: 0.8156305506216697
Test F1 for mss 20 : 0.609375
----------------------------------------
Train F1 for mss 50: 0.7427113702623906
Test F1 for mss 50 : 0.6082191780821918
----------------------------------------
Train F1 for mss 100: 0.7153179190751445
Test F1 for mss 100 : 0.6149659863945578
----------------------------------------


In [28]:
leaf_values = [1, 2, 5, 10, 20, 50]

for lv in leaf_values:
    dt = DecisionTreeClassifier(random_state=42,min_samples_leaf=lv)
    dt.fit(X_train_final,y_train)

    y_train_pred = dt.predict(X_train_final)
    y_test_pred = dt.predict(X_test_final)

    train_f1 = f1_score(y_train,y_train_pred)
    test_f1 = f1_score(y_test,y_test_pred)

    print(f"Train F1 for msl {lv}: {train_f1}")
    print(f"Test F1 for msl {lv} : {test_f1}")
    print("----------------------------------------")
    

Train F1 for msl 1: 1.0
Test F1 for msl 1 : 0.5493975903614458
----------------------------------------
Train F1 for msl 2: 0.9279216235129462
Test F1 for msl 2 : 0.5581395348837209
----------------------------------------
Train F1 for msl 5: 0.8328651685393258
Test F1 for msl 5 : 0.5974358974358974
----------------------------------------
Train F1 for msl 10: 0.7526334907373774
Test F1 for msl 10 : 0.6145552560646901
----------------------------------------
Train F1 for msl 20: 0.7134587554269175
Test F1 for msl 20 : 0.6335570469798658
----------------------------------------
Train F1 for msl 50: 0.6739705326785039
Test F1 for msl 50 : 0.6221590909090909
----------------------------------------


In [31]:

    dt = DecisionTreeClassifier(random_state=42,min_samples_leaf=20,min_samples_split=100,max_depth=4)
    dt.fit(X_train_final,y_train)

    y_train_pred = dt.predict(X_train_final)
    y_test_pred = dt.predict(X_test_final)

    train_f1 = f1_score(y_train,y_train_pred)
    test_f1 = f1_score(y_test,y_test_pred)

    print(f"Train F1 : {train_f1}")
    print(f"Test F1 : {test_f1}")
    print("----------------------------------------")
    

Train F1 : 0.6676374226428832
Test F1 : 0.648
----------------------------------------


In [32]:
from sklearn.model_selection import GridSearchCV
dt = DecisionTreeClassifier(random_state = 42)

param_grid = {
    'max_depth': [3, 4, 5, 6, 7],
    'min_samples_split': [10, 20, 50, 100],
    'min_samples_leaf': [5, 10, 20, 50]
}

grid_search = GridSearchCV(
    estimator=dt,
    param_grid=param_grid,
    scoring='f1',
    cv=5,
    n_jobs=-1
)
grid_search.fit(X_train_final, y_train)
print("Best Parameters:", grid_search.best_params_)
print("Best CV F1:", grid_search.best_score_)

Best Parameters: {'max_depth': 3, 'min_samples_leaf': 5, 'min_samples_split': 10}
Best CV F1: 0.6478927299093006


In [34]:
best_dt = grid_search.best_estimator_
y_test_pred = best_dt.predict(X_test_final)

test_f1 = f1_score(y_test, y_test_pred)

print("Final Test F1:", test_f1)

Final Test F1: 0.6357615894039735


# Post_Pruning

In [35]:
dt = DecisionTreeClassifier(random_state=42)
dt.fit(X_train_final,y_train)

path = dt.cost_complexity_pruning_path(X_train_final, y_train)

ccp_alphas = path.ccp_alphas

print(ccp_alphas)

[0.00000000e+00 5.05309712e-05 6.57591898e-05 6.67907066e-05
 6.68348803e-05 6.69657805e-05 6.69851979e-05 7.60340633e-05
 8.68960723e-05 8.68960723e-05 8.68960723e-05 8.96812028e-05
 9.01144453e-05 9.01144453e-05 9.01144453e-05 9.12408759e-05
 9.21602868e-05 9.21625009e-05 9.21625009e-05 9.21625009e-05
 9.25632074e-05 9.25632074e-05 9.29305218e-05 9.29305218e-05
 9.32684509e-05 9.35803856e-05 9.35803856e-05 9.35803856e-05
 9.40135947e-05 9.41374117e-05 9.46201676e-05 9.48381864e-05
 9.50425791e-05 9.50425791e-05 9.50425791e-05 9.54152951e-05
 9.54152951e-05 9.54152951e-05 9.54152951e-05 9.57465982e-05
 9.57465982e-05 9.60430273e-05 9.60430273e-05 9.61798407e-05
 9.71546364e-05 9.74795683e-05 9.74795683e-05 9.74795683e-05
 9.75888912e-05 9.77580813e-05 9.78829320e-05 9.79633172e-05
 9.83066676e-05 9.90211056e-05 9.90211056e-05 9.91258899e-05
 9.91748651e-05 9.92744302e-05 9.96001764e-05 9.98110384e-05
 9.98190779e-05 9.98740185e-05 1.00656135e-04 1.00876876e-04
 1.01378751e-04 1.013787

In [36]:
print(len(ccp_alphas))

319


In [37]:
print(ccp_alphas[:20])

[0.00000000e+00 5.05309712e-05 6.57591898e-05 6.67907066e-05
 6.68348803e-05 6.69657805e-05 6.69851979e-05 7.60340633e-05
 8.68960723e-05 8.68960723e-05 8.68960723e-05 8.96812028e-05
 9.01144453e-05 9.01144453e-05 9.01144453e-05 9.12408759e-05
 9.21602868e-05 9.21625009e-05 9.21625009e-05 9.21625009e-05]


In [38]:
train_f1 = []
test_f1 = []

for alpha in ccp_alphas:
    dt = DecisionTreeClassifier(
        random_state=42,
        ccp_alpha=alpha
    )

    dt.fit(X_train_final, y_train)

    y_train_pred = dt.predict(X_train_final)
    y_test_pred = dt.predict(X_test_final)

    train_f1.append(f1_score(y_train, y_train_pred))
    test_f1.append(f1_score(y_test, y_test_pred))

In [40]:
print("Number of alphas:", len(ccp_alphas))

for alpha, train, test in zip(ccp_alphas, train_f1, test_f1):
    print(f"alpha={alpha:.8f} | Train F1={train:.4f} | Test F1={test:.4f}")

Number of alphas: 319
alpha=0.00000000 | Train F1=1.0000 | Test F1=0.5494
alpha=0.00005053 | Train F1=0.9997 | Test F1=0.5494
alpha=0.00006576 | Train F1=0.9993 | Test F1=0.5494
alpha=0.00006679 | Train F1=0.9990 | Test F1=0.5494
alpha=0.00006683 | Train F1=0.9987 | Test F1=0.5494
alpha=0.00006697 | Train F1=0.9983 | Test F1=0.5494
alpha=0.00006699 | Train F1=0.9980 | Test F1=0.5494
alpha=0.00007603 | Train F1=0.9977 | Test F1=0.5494
alpha=0.00008690 | Train F1=0.9967 | Test F1=0.5507
alpha=0.00008690 | Train F1=0.9967 | Test F1=0.5507
alpha=0.00008690 | Train F1=0.9967 | Test F1=0.5507
alpha=0.00008968 | Train F1=0.9957 | Test F1=0.5511
alpha=0.00009011 | Train F1=0.9946 | Test F1=0.5518
alpha=0.00009011 | Train F1=0.9946 | Test F1=0.5518
alpha=0.00009011 | Train F1=0.9946 | Test F1=0.5518
alpha=0.00009124 | Train F1=0.9940 | Test F1=0.5525
alpha=0.00009216 | Train F1=0.9933 | Test F1=0.5525
alpha=0.00009216 | Train F1=0.9923 | Test F1=0.5525
alpha=0.00009216 | Train F1=0.9923 | Test 

In [41]:
from sklearn.model_selection import GridSearchCV

dt = DecisionTreeClassifier(random_state=42)

param_grid = {
    'ccp_alpha': ccp_alphas
}

grid_ccp = GridSearchCV(
    estimator=dt,
    param_grid=param_grid,
    scoring='f1',
    cv=5,
    n_jobs=-1
)

grid_ccp.fit(X_train_final, y_train)

print("Best ccp_alpha:", grid_ccp.best_params_)
print("Best CV F1:", grid_ccp.best_score_)

best_ccp_dt = grid_ccp.best_estimator_

y_test_pred = best_ccp_dt.predict(X_test_final)

print("Final Test F1:", f1_score(y_test, y_test_pred))

Best ccp_alpha: {'ccp_alpha': np.float64(0.001906013423500913)}
Best CV F1: 0.6340460779511914
Final Test F1: 0.6258692628650904
